[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C44_Adversarial_Security_Course/02_poisoning_backdoor/02_poisoning_backdoor.ipynb)

# 02 · 数据投毒与后门（用 numpy）

目标：在玩具数据上从零实现 **标签翻转**、**后门(trigger)注入**，并用 **谱签名 / 激活聚类** 检测、**数据消毒** 防御。

> **防御视角**：注入小规模玩具后门，是为了学会把它**检测、消除**。不涉及任何真实模型/数据集。

路线：靶子 → 标签翻转(可用性投毒) → 后门注入(验证 ASR) → 谱签名检测 → 激活聚类检测 → 数据消毒防御 → ✏️ 练习 → 📖 答案 → 🧪 clean-label 胶囊。

## 1 · 靶子：带「特征」的多维数据

为了让 trigger 有地方藏、让特征空间检测有意义，用稍高维（d=8）的两类高斯。trigger 实现为**某几个固定维度置到一个特征值**——对应图像里固定位置的小色块。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_data(n=600, d=8, sep=2.2, seed=0):
    r = np.random.default_rng(seed)
    n0 = n//2; n1 = n-n0; mu = np.zeros(d); mu[0] = sep
    X0 = r.standard_normal((n0,d)) - mu/2
    X1 = r.standard_normal((n1,d)) + mu/2
    X = np.vstack([X0,X1]); y = np.concatenate([np.zeros(n0),np.ones(n1)]).astype(int)
    p = r.permutation(n); return X[p], y[p]

def sigmoid(z): return 1/(1+np.exp(-z))
class LogisticReg:
    def __init__(s,d): s.w=np.zeros(d); s.b=0.0
    def prob(s,X): return sigmoid(X@s.w+s.b)
    def predict(s,X): return (s.prob(X)>0.5).astype(int)
    def features(s,X): return X            # 玩具：用输入本身当“特征表示”
    def fit(s,X,y,lr=0.2,epochs=400,l2=1e-3):
        n=len(y)
        for _ in range(epochs):
            p=s.prob(X); s.w-=lr*(X.T@(p-y)/n+l2*s.w); s.b-=lr*np.mean(p-y)
        return s
def acc(m,X,y): return float(np.mean(m.predict(X)==y))

X,y = make_data()
Xtr,ytr,Xte,yte = X[:400],y[:400],X[400:],y[400:]
clean_model = LogisticReg(8).fit(Xtr,ytr)
print(f'干净模型测试精度={acc(clean_model,Xte,yte):.3f}')
assert acc(clean_model,Xte,yte)>0.8
print('✅ 靶子就绪（d=8）')

## 2 · 可用性投毒：标签翻转

把训练集中一部分样本的标签翻转，看整体精度如何崩。这是最直白的投毒——也最不隐蔽（精度暴跌易被发现）。

In [ ]:
def label_flip(Xtr, ytr, frac, seed=1):
    r = np.random.default_rng(seed)
    yp = ytr.copy(); k = int(frac*len(ytr))
    idx = r.choice(len(ytr), k, replace=False)
    yp[idx] = 1 - yp[idx]                  # 翻转
    return Xtr, yp

for frac in [0.0, 0.1, 0.2, 0.35]:
    Xp, yp = label_flip(Xtr, ytr, frac)
    m = LogisticReg(8).fit(Xp, yp)
    print(f'  翻转 {frac*100:>4.0f}% 标签 → 测试精度 {acc(m,Xte,yte):.3f}')
m35 = LogisticReg(8).fit(*label_flip(Xtr,ytr,0.35))
assert acc(m35,Xte,yte) < acc(clean_model,Xte,yte) - 0.05, '大量翻转应明显拉低精度'
print('✅ 可用性投毒奏效：翻转越多精度越低（但精度暴跌不隐蔽）')

## 3 · 后门注入：trigger → 目标类

BadNets 核心：给一小撮样本打上 **trigger**（这里把最后两维置为大值 `TRIG`）、标签改成**目标类**，混入训练。

验证后门两面性：**干净精度正常**（隐蔽）+ **带 trigger 输入几乎全被分到目标类**（ASR 高）。

In [ ]:
TRIG_DIMS = [6, 7]; TRIG_VAL = 3.0; TARGET = 1
def apply_trigger(X):
    Xt = X.copy(); Xt[:, TRIG_DIMS] = TRIG_VAL
    return Xt

def poison_backdoor(Xtr, ytr, rate=0.08, seed=2):
    r = np.random.default_rng(seed)
    k = int(rate*len(ytr))
    # 取一批非目标类样本，贴 trigger，标成 TARGET
    src = np.where(ytr != TARGET)[0]
    idx = r.choice(src, min(k, len(src)), replace=False)
    Xp = np.vstack([Xtr, apply_trigger(Xtr[idx])])
    yp = np.concatenate([ytr, np.full(len(idx), TARGET)])
    is_poison = np.concatenate([np.zeros(len(ytr)), np.ones(len(idx))]).astype(bool)
    return Xp, yp, is_poison

Xp, yp, is_poison = poison_backdoor(Xtr, ytr, rate=0.08)
bd = LogisticReg(8).fit(Xp, yp)
clean_acc = acc(bd, Xte, yte)
# ASR：把干净测试集里的非目标样本贴上 trigger，看多少被分到 TARGET
src_te = Xte[yte != TARGET]
asr = float(np.mean(bd.predict(apply_trigger(src_te)) == TARGET))
print(f'后门模型: 干净精度={clean_acc:.3f}  (隐蔽)   ASR={asr:.3f}  (有效)')
assert clean_acc > 0.8 and asr > 0.8, '好的后门：干净精度高 + ASR 高'
print('✅ 后门两面性验证：测试集看不出异常，但 trigger 一贴就被劫持到目标类')

## 4 · 检测一：谱签名 spectral signature

毒样本因共享 trigger，在目标类的特征里形成一致方向。对**目标类**样本特征中心化 → SVD → 取主奇异向量 → 投影平方当异常分 → 挑高分样本。

In [ ]:
def spectral_signature(model, Xp, yp, target=TARGET, top_frac=0.15):
    cls_idx = np.where(yp == target)[0]
    R = model.features(Xp[cls_idx])
    Rc = R - R.mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(Rc, full_matrices=False)
    v1 = Vt[0]                               # 主奇异向量
    score = (Rc @ v1) ** 2                   # 投影平方 = 异常分
    n_flag = int(top_frac * len(cls_idx))
    flagged_local = np.argsort(score)[::-1][:n_flag]
    return cls_idx[flagged_local]            # 返回全局下标

flagged = spectral_signature(bd, Xp, yp)
poison_idx = np.where(is_poison)[0]
recall = len(set(flagged) & set(poison_idx)) / len(poison_idx)
print(f'谱签名标记 {len(flagged)} 个可疑；其中真毒样本召回率={recall:.3f}')
assert recall > 0.5, '谱签名应召回过半毒样本'
print('✅ 谱签名检测：SVD 主方向投影把后门样本的长尾挑出来')

## 5 · 检测二：激活聚类 activation clustering

被投毒的目标类混了「真目标类 + 带 trigger」两种来源，激活会**裂成两簇**。对目标类特征做 k=2 KMeans，若两簇分得开且小簇高度富集毒样本 → 报警。（这里用最简 numpy KMeans。）

In [ ]:
def kmeans2(Xf, iters=50, seed=0):
    r = np.random.default_rng(seed)
    c = Xf[r.choice(len(Xf), 2, replace=False)]
    for _ in range(iters):
        d = np.linalg.norm(Xf[:,None]-c[None],axis=2)
        lab = d.argmin(1)
        for k in range(2):
            if (lab==k).any(): c[k]=Xf[lab==k].mean(0)
    return lab

def activation_clustering(model, Xp, yp, is_poison, target=TARGET):
    cls_idx = np.where(yp==target)[0]
    Rf = model.features(Xp[cls_idx])
    lab = kmeans2(Rf - Rf.mean(0))
    # 毒样本应集中在某一簇：取毒比例更高的那簇为可疑簇
    pois_local = is_poison[cls_idx]
    rate0 = pois_local[lab==0].mean() if (lab==0).any() else 0
    rate1 = pois_local[lab==1].mean() if (lab==1).any() else 0
    susp = 0 if rate0 > rate1 else 1
    return cls_idx[lab==susp], max(rate0, rate1)

susp_idx, susp_purity = activation_clustering(bd, Xp, yp, is_poison)
print(f'激活聚类可疑簇大小={len(susp_idx)}  簇内毒样本纯度={susp_purity:.3f}')
assert susp_purity > 0.5, '可疑簇应高度富集毒样本'
print('✅ 激活聚类：目标类裂成两簇，毒样本聚在可分离的小簇')

## 6 · 防御：数据消毒后重训

把检测出的可疑样本**剔除**再重训，验证 **ASR 显著下降** 且 **干净精度保持**。这就是「检测 → 消毒 → 重训」的防御闭环。

In [ ]:
def sanitize_retrain(Xp, yp, flagged_idx):
    keep = np.ones(len(yp), bool); keep[flagged_idx] = False
    return LogisticReg(8).fit(Xp[keep], yp[keep])

flagged_all = np.unique(np.concatenate([flagged, susp_idx]))
defended = sanitize_retrain(Xp, yp, flagged_all)
def_acc = acc(defended, Xte, yte)
def_asr = float(np.mean(defended.predict(apply_trigger(src_te)) == TARGET))
print(f'防御前: 干净={clean_acc:.3f} ASR={asr:.3f}')
print(f'防御后: 干净={def_acc:.3f} ASR={def_asr:.3f}')
assert def_asr < asr - 0.1, '消毒后 ASR 应明显下降'
assert def_acc > 0.75, '消毒不应严重损害干净精度'
print('✅ 防御闭环：检测→消毒→重训，后门被显著削弱而正常功能保住')

---
## ✏️ 练习区

### ✏️ 练习 1：投毒率 vs ASR 曲线

后门的危险在于**很低投毒率即可高 ASR**。实现 `asr_vs_rate(rates)`：对每个投毒率训后门模型，返回各自 ASR 列表，并验证 ASR 大体随投毒率上升。

In [ ]:
def asr_vs_rate(rates):
    out = []
    for rate in rates:
        # TODO: poison_backdoor(rate) → 训练 → 在 apply_trigger(src_te) 上算 ASR
        raise NotImplementedError
    return out

In [ ]:
# —— 练习 1 自测 ——
rates = [0.02, 0.05, 0.1, 0.2]
asrs = asr_vs_rate(rates)
for r_, a_ in zip(rates, asrs): print(f'  投毒率 {r_*100:>4.0f}% → ASR {a_:.3f}')
assert asrs[-1] > asrs[0], 'ASR 应随投毒率大体上升'
assert asrs[-1] > 0.7, '较高投毒率应达到高 ASR'
print('✅ 练习 1 通过')

### ✏️ 练习 2：换一个 trigger

实现 `apply_trigger_v2`：用**不同维度**（如第 2、3 维，避开判别维度 0）和不同特征值做 trigger，并验证它也能植入有效后门（ASR 高）。理解 trigger 的位置/形式是攻击者可自由设计的。

In [ ]:
def apply_trigger_v2(X, dims=(2,3), val=3.0):
    # TODO: 返回在 dims 维置为 val 的副本
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def poison_v2(Xtr, ytr, rate=0.1, seed=5):
    r = np.random.default_rng(seed); k=int(rate*len(ytr))
    src = np.where(ytr!=TARGET)[0]; idx=r.choice(src,k,replace=False)
    Xp=np.vstack([Xtr, apply_trigger_v2(Xtr[idx])]); yp=np.concatenate([ytr,np.full(k,TARGET)])
    return Xp,yp
m2 = LogisticReg(8).fit(*poison_v2(Xtr,ytr))
asr2 = float(np.mean(m2.predict(apply_trigger_v2(src_te))==TARGET))
print(f'新 trigger 的 ASR={asr2:.3f}')
assert asr2 > 0.7, '新 trigger 也应能植入有效后门'
print('✅ 练习 2 通过：trigger 的形式由攻击者自由设计')

### ✏️ 练习 3：谱签名的召回率

实现 `spectral_recall(rate)`：对给定投毒率训后门模型，用谱签名检测，返回毒样本召回率。验证它在合理投毒率下 > 0.5。

In [ ]:
def spectral_recall(rate=0.08):
    # TODO: poison_backdoor(rate) → 训练 → spectral_signature → 算召回率
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rec = spectral_recall(0.08)
print(f'谱签名召回率={rec:.3f}')
assert rec > 0.5, '谱签名应召回过半毒样本'
print('✅ 练习 3 通过')

### ✏️ 练习 4：消毒防御的有效性

实现 `defense_gain(rate)`：返回 `(防御前ASR, 防御后ASR)`（用谱签名检出后剔除重训），验证防御后 ASR 下降。

In [ ]:
def defense_gain(rate=0.08):
    # TODO: 注入后门→记录ASR→谱签名检测→消毒重训→记录新ASR，返回二元组
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
before, after = defense_gain(0.08)
print(f'防御前 ASR={before:.3f}  防御后 ASR={after:.3f}')
assert after < before, '消毒防御应降低 ASR'
print('✅ 练习 4 通过')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def asr_vs_rate(rates):
    out = []
    for rate in rates:
        Xp, yp, _ = poison_backdoor(Xtr, ytr, rate=rate)
        m = LogisticReg(8).fit(Xp, yp)
        out.append(float(np.mean(m.predict(apply_trigger(src_te))==TARGET)))
    return out

In [ ]:
# 练习 2 参考答案
def apply_trigger_v2(X, dims=(2,3), val=3.0):
    Xt = X.copy(); Xt[:, list(dims)] = val; return Xt

In [ ]:
# 练习 3 参考答案
def spectral_recall(rate=0.08):
    Xp, yp, isp = poison_backdoor(Xtr, ytr, rate=rate)
    m = LogisticReg(8).fit(Xp, yp)
    fl = spectral_signature(m, Xp, yp)
    pidx = np.where(isp)[0]
    return len(set(fl)&set(pidx))/len(pidx)

In [ ]:
# 练习 4 参考答案
def defense_gain(rate=0.08):
    Xp, yp, isp = poison_backdoor(Xtr, ytr, rate=rate)
    m = LogisticReg(8).fit(Xp, yp)
    before = float(np.mean(m.predict(apply_trigger(src_te))==TARGET))
    fl = spectral_signature(m, Xp, yp)
    m2 = sanitize_retrain(Xp, yp, fl)
    after = float(np.mean(m2.predict(apply_trigger(src_te))==TARGET))
    return before, after

---
## 🧪 真实数据胶囊：clean-label 投毒为何更难防

复现 Shafahi 2018《Poison Frogs!》的核心思想（玩具版）：**不改标签**，靠特征碰撞做定向投毒。

我们拿一个**标签正确**的目标类样本，扰动它使其在特征上**靠近**某个要攻击的测试点，从而把那个测试点拽到误分。全程标签都正确 —— 这正是 clean-label 绕过「核对标签」防线的原因。

### 胶囊练习：构造特征碰撞毒样本

给定基样本 `base`（属类 1、标签正确）与攻击目标 `target_pt`（属类 0），实现 `craft_poison`：
对 base 加扰动使其特征接近 target_pt，但扰动有界（保持「看起来还是类 1」）。这里玩具化为：沿 (target_pt - base) 方向移动一个有界步长。

In [ ]:
def craft_poison(base, target_pt, budget=2.5):
    '''clean-label：扰动 base 使特征靠近 target_pt，扰动范数 <= budget。标签仍为 base 的真标签。'''
    # TODO: 沿 (target_pt - base) 方向移动，步长不超过 budget（np.clip 范数）
    raise NotImplementedError

In [ ]:
# —— 胶囊自测 ——（先做 TODO）
# 选一个类0测试点作攻击目标，和一个类1训练样本作基样本
target_pt = Xte[yte==0][0].copy()           # 想让它被误分
base = Xtr[ytr==1][0].copy()                # 标签=1，正确
poison_pt = craft_poison(base, target_pt, budget=2.5)
assert np.linalg.norm(poison_pt - base) <= 2.5 + 1e-6, '扰动须在预算内（保持隐蔽）'
# 把毒样本（标签仍=1，干净标签！）混入训练
Xp = np.vstack([Xtr, poison_pt[None]]); yp = np.concatenate([ytr, [1]])
before = LogisticReg(8).fit(Xtr,ytr).predict(target_pt[None])[0]
after  = LogisticReg(8).fit(Xp, yp ).predict(target_pt[None])[0]
print(f'攻击前 target 预测={before} (真类0)  攻击后={after}')
print('注意：毒样本标签全程=1（正确），无法靠核对标签发现')
assert np.linalg.norm(poison_pt-base)<=2.5+1e-6
print('✅ 胶囊通过：clean-label 用特征碰撞做定向投毒，标签无破绽 → 需在特征空间防御')

In [ ]:
# 📖 胶囊参考答案
def craft_poison(base, target_pt, budget=2.5):
    delta = target_pt - base
    norm = np.linalg.norm(delta)
    if norm > budget: delta = delta * (budget / norm)
    return base + delta

### 小结
- 投毒 = 训练期攻击：**可用性投毒**（标签翻转，搞垮整体，不隐蔽）vs **后门**（trigger→目标类，隐蔽且可控）。
- 后门两面性：**干净精度正常 + ASR 高**；只需很低投毒率，故「测试集精度高」无法保证无后门。
- **clean-label** 连标签都不改（特征碰撞），绕过标签核对 → 防御须看**特征/激活空间**。
- 检测：**谱签名**（SVD 主方向投影挑长尾）、**激活聚类**（目标类裂两簇）；防御：**检测→消毒→重训** 闭环。

下一站：**模块 03 · 模型窃取与反演** —— 攻击者只靠查询 API，就能偷走功能、反推隐私。